# 08 — Extraordinary events (exploratory)
 **RQ9:** How did bike-sharing rental counts change around an earthquake and Hurricane Sandy?

 These dates are event annotations, not experimental interventions. The hurricane also affected service availability. Baselines are illustrative comparisons with nearby days of the **same weekday**; do not treat as counterfactual forecasts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

day=pd.read_csv('../data/processed/bike_day_clean.csv')
day['dteday']=pd.to_datetime(day['dteday'])
events={'2011-08-23':'Virginia Earthquake',
        '2012-10-29':'Hurricane Sandy'}

fig,axes=plt.subplots(2,1,figsize=(12.5,8.2))
for ax,(date,name) in zip(axes,events.items()):
    t=pd.Timestamp(date)
    window=day.loc[day['dteday'].between(t-pd.Timedelta(days=14),
                                         t+pd.Timedelta(days=14))]
    event_rentals=int(day.loc[day['dteday']==t,'cnt'].iloc[0])
    nearby=(day['dteday'].between(t-pd.Timedelta(days=28),
                                 t+pd.Timedelta(days=28))
            & day['dteday'].dt.day_of_week.eq(t.dayofweek)
            & day['dteday'].ne(t))
    reference=day.loc[nearby,'cnt'].median()
    delta=(event_rentals/reference-1)*100

    ax.plot(window['dteday'],window['cnt'],color='#2563EB',marker='o',markersize=3,
            linewidth=1.9,label='Daily rentals')
    ax.axvline(t,color='#DC2626',linestyle='--',linewidth=1.8,label='Event')
    ax.axhline(reference,color='#64748B',linestyle=':',linewidth=1.8,
               label='Nearby same-weekday median')
    ax.scatter(t,event_rentals,color='#DC2626',s=50,zorder=5)
    ax.annotate(f'{event_rentals:,} rentals\n({delta:+.1f}% vs reference)',
                xy=(t,event_rentals),xytext=(12,-37) if event_rentals > reference else (12,28),textcoords='offset points',
                fontsize=9,arrowprops={'arrowstyle':'->','color':'#555555'})
    ax.set_title(f'{name} — {t:%d %b %Y}',fontweight='bold')
    ax.set_ylabel('Daily rentals')
    ax.grid(axis='y',alpha=.17)
    ax.legend(frameon=False,fontsize=8,loc='upper left')
    ax.tick_params(axis='x',rotation=25)
    ax.set_ylim(bottom=0)
    print(f'{name}: {event_rentals:,} rentals; reference {reference:,.1f}; '
          f'descriptive difference {delta:+.1f}%')

fig.suptitle('Exceptional events coincided with unusual rental counts',
             fontsize=14,fontweight='bold',y=1.01)
fig.text(.01,-.025,'Descriptive comparison only: days of week and short windows are controlled, '
         'but weather, closures and other factors are not. Source: UCI / Capital Bikeshare.',
         fontsize=9,color='#555555')
fig.tight_layout()
Path('../output/figures').mkdir(parents=True,exist_ok=True)
fig.savefig('../output/figures/RQ9_extraordinary_events.png',dpi=200,bbox_inches='tight')
plt.show()

### Evidence and caution
 - 23 August 2011: Virginia earthquake — external historical event, verify transportation disruption details from dated reports.
 - 29 October 2012: Hurricane Sandy — transportation and bike-sharing services were affected. The change in rental counts reflects *both availability and use*.
 - A difference from nearby Tuesdays/Mondays is **not an estimate of a causal event effect**. Cite independent historical sources on the final slide.